# Experiment 13 - XGBoost Subsample Tuning

This experiment tests the `subsample` parameter using the current best XGBoost configuration.

The current best validation ROC-AUC is **0.941747** from Experiment 11.

Only `subsample` will be changed. All other model settings and preprocessing will stay the same.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

train = pd.read_csv(r"C:\Users\aakif\Documents\DataCompetition\data\train.csv")

X = train.drop(columns=["Will_Buy_EV", "id"])
y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

print("Train shape:", X_train_processed.shape)
print("Validation shape:", X_valid_processed.shape)


Train shape: (534932, 24)
Validation shape: (133733, 24)


In [2]:
configs = {
    "S1_subsample_0.80": 0.80,
    "S2_subsample_0.90": 0.90,
}

results = []

for name, subsample in configs.items():
    print("=" * 60)
    print(name)
    print("=" * 60)

    model = XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=2,
        subsample=subsample,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_processed, y_train)

    valid_predictions = model.predict_proba(X_valid_processed)[:, 1]
    score = roc_auc_score(y_valid, valid_predictions)

    results.append({
        "Experiment": name,
        "subsample": subsample,
        "ROC-AUC": score
    })

    print(f"subsample: {subsample}")
    print(f"ROC-AUC: {score:.6f}")
    print()


S1_subsample_0.80
subsample: 0.8
ROC-AUC: 0.941773

S2_subsample_0.90
subsample: 0.9
ROC-AUC: 0.941776



In [3]:
results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

best_result = results_df.iloc[0]
best_score = best_result["ROC-AUC"]
previous_best = 0.941747
difference = best_score - previous_best

print("=" * 60)
print("EXPERIMENT 13 RESULTS")
print("=" * 60)
print(results_df.to_string(index=False))
print()
print(f"Previous best ROC-AUC: {previous_best:.6f}")
print(f"Best Experiment 13 ROC-AUC: {best_score:.6f}")
print(f"Difference: {difference:+.6f}")
print()

if best_score > previous_best:
    print("NEW BEST MODEL")
else:
    print("No improvement over the current best model")


EXPERIMENT 13 RESULTS
       Experiment  subsample  ROC-AUC
S2_subsample_0.90        0.9 0.941776
S1_subsample_0.80        0.8 0.941773

Previous best ROC-AUC: 0.941747
Best Experiment 13 ROC-AUC: 0.941776
Difference: +0.000029

NEW BEST MODEL


## Result

The best configuration from this experiment is compared against the previous best ROC-AUC of **0.941747**.

If the score improves, this configuration becomes the new candidate for the next submission.
